# Procesos Estocásticos — Ejemplos y Simulaciones

Este notebook acompaña los apuntes teóricos con ejemplos concretos, simulaciones y visualizaciones.  
Cada módulo asume que la definición formal ya fue presentada en clase.

**Referencias generales:**  
- Karatzas, I. & Shreve, S. (1991). *Brownian Motion and Stochastic Calculus*. Springer.  
- Norris, J. (1997). *Markov Chains*. Cambridge University Press.  
- Billingsley, P. (1995). *Probability and Measure*. Wiley.  
- Durrett, R. (2019). *Probability: Theory and Examples*. Cambridge University Press.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.linalg import expm

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("Librerías cargadas.")

---
# Módulo 0 — Espacio de probabilidad y procesos estocásticos

Un **proceso estocástico** es una familia $\{X_t\}_{t \in T}$ de variables aleatorias definidas sobre un espacio de probabilidad $(\Omega, \mathcal{F}, P)$.  
Para cada $\omega \in \Omega$ fijo, la función $t \mapsto X_t(\omega)$ es una **trayectoria** (o realización) del proceso.

Una **filtración** $\{\mathcal{F}_t\}$ es una familia creciente de sub-$\sigma$-álgebras de $\mathcal{F}$: representa la información disponible hasta el tiempo $t$.

### Ejemplo 0.1 — Filtración por lanzamiento de monedas

Lanzamos una moneda justa tres veces. El espacio muestral es $\Omega = \{C, S\}^3$ (8 elementos).  
La filtración natural es:
- $\mathcal{F}_0 = \{\emptyset, \Omega\}$ — no sabemos nada
- $\mathcal{F}_1$ — sabemos el resultado del primer lanzamiento (2 átomos)
- $\mathcal{F}_2$ — sabemos los dos primeros (4 átomos)
- $\mathcal{F}_3 = \mathcal{F}$ — sabemos todo (8 átomos)

Sea $X_t = $ número de caras hasta el tiempo $t$. Este proceso es **adaptado** a $\{\mathcal{F}_t\}$.

In [ ]:
# Simulamos todas las trayectorias posibles del proceso X_t = # caras hasta t
from itertools import product

omega = list(product([0, 1], repeat=3))  # 0=seca, 1=cara
trayectorias = np.array([[0, r[0], r[0]+r[1], r[0]+r[1]+r[2]] for r in omega])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: todas las trayectorias
colores = plt.cm.tab10(np.linspace(0, 0.8, len(omega)))
for i, (tray, om) in enumerate(zip(trayectorias, omega)):
    label = ''.join(['C' if x else 'S' for x in om])
    axes[0].step(range(4), tray, where='post', color=colores[i], lw=1.8,
                 alpha=0.85, label=label)
axes[0].set_xticks(range(4))
axes[0].set_xlabel('t  (lanzamiento)')
axes[0].set_ylabel('X_t  (caras acumuladas)')
axes[0].set_title('Todas las trayectorias de X_t')
axes[0].legend(fontsize=8, ncol=2, title='ω')

# Panel derecho: átomos de la filtración en t=1 y t=2
ax = axes[1]
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 8.5)
ax.set_title('Átomos de la filtración  $\\mathcal{F}_0 \\subset \\mathcal{F}_1 \\subset \\mathcal{F}_2 \\subset \\mathcal{F}_3$')
ax.set_xlabel('t')
ax.set_yticks([])
ax.grid(False)

etiquetas = ['SSS','SSC','SCS','SCC','CSS','CSC','CCS','CCC']
for i, et in enumerate(etiquetas):
    ax.text(3.1, i, et, va='center', fontsize=9)

# t=0: un solo átomo
ax.add_patch(plt.Rectangle((-0.4, -0.4), 0.8, 8.8, color='#2196F3', alpha=0.15))
ax.text(0, 4, 'Ω', ha='center', va='center', fontsize=13, color='#2196F3')

# t=1: dos átomos (primer lanzamiento S o C)
for rango, color, label in [((-.4,3.6),(4,.4),'#4CAF50','S??'),
                              ((3.6,7.6),(4,4.4),'#E91E63','C??')]:
    pass
ax.add_patch(plt.Rectangle((0.7, -0.4), 0.6, 4.0, color='#4CAF50', alpha=0.15))
ax.add_patch(plt.Rectangle((0.7,  3.6), 0.6, 4.8, color='#E91E63', alpha=0.15))
ax.text(1, 1.5, 'S??', ha='center', va='center', fontsize=10, color='#4CAF50')
ax.text(1, 5.9, 'C??', ha='center', va='center', fontsize=10, color='#E91E63')

# t=2: cuatro átomos
for (y0, h, color, label) in [(-.4,2,'#1565C0','SS?'),(1.6,2,'#388E3C','SC?'),
                               (3.6,2,'#C62828','CS?'),(5.6,2,'#6A1B9A','CC?')]:
    ax.add_patch(plt.Rectangle((1.7, y0), 0.6, h, color=color, alpha=0.15))
    ax.text(2, y0 + h/2, label, ha='center', va='center', fontsize=9, color=color)

ax.set_xticks([0,1,2,3])
plt.tight_layout()
plt.show()

### Ejemplo 0.2 — Trayectorias vs la función $X(\omega, t)$

Un proceso estocástico es simultáneamente:
- Para cada $t$ fijo: una **variable aleatoria** $X_t : \Omega \to \mathbb{R}$
- Para cada $\omega$ fijo: una **función determinista** $t \mapsto X_t(\omega)$ (trayectoria)
- En conjunto: una **función medible** $X : \Omega \times T \to \mathbb{R}$

El siguiente ejemplo usa un proceso gaussiano simple para ilustrar ambas perspectivas.

In [ ]:
# Proceso: X_t = A * cos(t) + B * sin(t),  A,B ~ N(0,1) iid
# Media: E[X_t] = 0
# Covarianza: Cov(X_s, X_t) = cos(s-t)  (proceso estacionario)

T = np.linspace(0, 4 * np.pi, 300)
n_tray = 6
t_corte = np.pi  # tiempo fijo para ver la distribución marginal

A = np.random.normal(0, 1, n_tray)
B = np.random.normal(0, 1, n_tray)
trayectorias_gp = np.outer(A, np.cos(T)) + np.outer(B, np.sin(T))

# Distribución marginal en t = π
A_mc = np.random.normal(0, 1, 5000)
B_mc = np.random.normal(0, 1, 5000)
X_en_t = A_mc * np.cos(t_corte) + B_mc * np.sin(t_corte)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trayectorias
for i in range(n_tray):
    axes[0].plot(T, trayectorias_gp[i], lw=1.5, alpha=0.75, label=f'ω_{i+1}')
axes[0].axvline(t_corte, color='black', lw=1.5, ls='--', label=f't = π')
axes[0].set_xlabel('t')
axes[0].set_ylabel('X_t(ω)')
axes[0].set_title('Perspectiva trayectoria: $t \\mapsto X_t(\\omega)$\n'
                   '$X_t = A\\cos(t) + B\\sin(t)$,   $A,B \\sim N(0,1)$')
axes[0].legend(fontsize=8)

# Distribución marginal en t = π
axes[1].hist(X_en_t, bins=50, density=True, color='#2196F3', alpha=0.6, label='Marginal empírica')
xx = np.linspace(-4, 4, 300)
axes[1].plot(xx, stats.norm.pdf(xx, 0, 1), 'r-', lw=2, label='N(0,1) teórica')
axes[1].set_xlabel('$X_{\\pi}(\\omega)$')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Perspectiva variable aleatoria: $\\omega \\mapsto X_{\\pi}(\\omega)$\n'
                   '(corte vertical en $t = \\pi$)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
# Módulo 1 — Estacionariedad y autocovarianza

**Estacionariedad estricta:** la distribución conjunta de $(X_{t_1}, \ldots, X_{t_k})$ es invariante a traslaciones en el tiempo.  
**Estacionariedad débil (de segundo orden):** $E[X_t] = \mu$ constante y $\text{Cov}(X_s, X_t) = \gamma(|s-t|)$ depende solo del lag.

La **función de autocovarianza** es $\gamma(h) = \text{Cov}(X_t, X_{t+h})$ y la **función de autocorrelación** (ACF) es $\rho(h) = \gamma(h)/\gamma(0)$.

In [ ]:
# Cuatro procesos: ruido blanco, MA(1), caminata aleatoria, con tendencia
n_obs = 300
theta_ma = 0.7  # parámetro MA(1)

eps = np.random.normal(0, 1, n_obs + 1)

ruido_blanco   = eps[1:]                                          # iid N(0,1)
ma1            = eps[1:] + theta_ma * eps[:-1]                    # MA(1)
caminata       = np.cumsum(np.random.normal(0, 1, n_obs))         # no estacionario
con_tendencia  = 0.05 * np.arange(n_obs) + np.random.normal(0, 1, n_obs)

procesos = [
    (ruido_blanco,  'Ruido blanco $W_t$\n(estrictamente estacionario)',    '#2196F3'),
    (ma1,           f'MA(1): $X_t = W_t + {theta_ma}W_{{t-1}}$\n(débilmente estacionario)', '#4CAF50'),
    (caminata,      'Caminata aleatoria $S_n = \\sum W_i$\n(NO estacionario)', '#E91E63'),
    (con_tendencia, 'Proceso con tendencia $0.05t + W_t$\n(NO estacionario)', '#FF9800'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))

for col, (proc, titulo, color) in enumerate(procesos):
    # Trayectoria
    axes[0, col].plot(proc, color=color, lw=1, alpha=0.85)
    axes[0, col].set_title(titulo, fontsize=9)
    axes[0, col].set_xlabel('t')

    # ACF empírica
    max_lag = 30
    acf_vals = [np.corrcoef(proc[:-lag], proc[lag:])[0, 1] if lag > 0 else 1.0
                for lag in range(max_lag + 1)]
    axes[1, col].bar(range(max_lag + 1), acf_vals, color=color, alpha=0.7, width=0.6)
    axes[1, col].axhline(0, color='black', lw=0.8)
    # Bandas de confianza aproximadas ±1.96/√n
    banda = 1.96 / np.sqrt(n_obs)
    axes[1, col].axhline( banda, color='gray', lw=1, ls='--')
    axes[1, col].axhline(-banda, color='gray', lw=1, ls='--')
    axes[1, col].set_xlabel('Lag h')
    axes[1, col].set_ylabel('ACF' if col == 0 else '')
    axes[1, col].set_ylim(-1.1, 1.1)

axes[0, 0].set_ylabel('X_t')
fig.suptitle('Trayectorias y ACF empírica — procesos estacionarios y no estacionarios',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Función de autocovarianza teórica vs empírica

Para el proceso MA(1): $X_t = W_t + \theta W_{t-1}$, $W_t \sim \text{iid}(0, \sigma^2)$:

$$\gamma(0) = (1+\theta^2)\sigma^2, \quad \gamma(1) = \theta\sigma^2, \quad \gamma(h) = 0 \text{ para } |h| > 1$$

**Referencia:** Box, G., Jenkins, G. & Reinsel, G. (2015). *Time Series Analysis: Forecasting and Control*. Wiley. Cap. 3.

In [ ]:
# ACF teórica vs empírica para MA(1)
sigma2 = 1.0
thetas = [0.3, 0.7, -0.7]
max_lag = 15
lags = np.arange(max_lag + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, theta in zip(axes, thetas):
    # Teórica
    acf_teo = np.zeros(max_lag + 1)
    acf_teo[0] = 1.0
    acf_teo[1] = theta / (1 + theta**2)

    # Empírica (muestra grande)
    n_sim = 2000
    eps_s = np.random.normal(0, sigma2**0.5, n_sim + 1)
    ma1_s = eps_s[1:] + theta * eps_s[:-1]
    acf_emp = [np.corrcoef(ma1_s[:-h], ma1_s[h:])[0, 1] if h > 0 else 1.0
               for h in lags]

    ax.bar(lags - 0.2, acf_teo, width=0.35, color='#2196F3', alpha=0.8, label='Teórica')
    ax.bar(lags + 0.2, acf_emp, width=0.35, color='#E91E63', alpha=0.6, label='Empírica')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'MA(1)  θ = {theta}', fontsize=11)
    ax.set_xlabel('Lag h')
    ax.set_ylabel('ACF')
    ax.set_ylim(-1.1, 1.1)
    ax.legend(fontsize=9)

fig.suptitle('ACF teórica vs empírica — proceso MA(1)\n'
             'Ref: Box, Jenkins & Reinsel (2015), Cap. 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo real — Precipitaciones mensuales en Buenos Aires

Las series de precipitación son un ejemplo clásico de proceso **estacionario con componente estacional**.  
La media y varianza son aproximadamente constantes en el tiempo, pero la ACF revela correlación a lags múltiplos de 12.

Simulamos una serie con esa estructura (SAR — proceso autorregresivo estacional):

$$X_t = \phi_{12} X_{t-12} + W_t$$

**Referencia:** Montanari, A. et al. (2000). *Fractionally differenced ARIMA models applied to hydrologic time series*. Water Resources Research, 33(5), 1035–1044.

In [ ]:
# SAR(1)_12: X_t = phi*X_{t-12} + W_t
phi12 = 0.65
n_meses = 240  # 20 años
w = np.random.normal(0, 1, n_meses)
x_sar = np.zeros(n_meses)
for t in range(12, n_meses):
    x_sar[t] = phi12 * x_sar[t - 12] + w[t]

# Media móvil anual para mostrar estacionariedad en media
media_movil = np.convolve(x_sar, np.ones(12)/12, mode='valid')

max_lag = 36
acf_sar = [np.corrcoef(x_sar[:-h], x_sar[h:])[0,1] if h > 0 else 1.0
           for h in range(max_lag + 1)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(x_sar, color='#00BCD4', lw=1, alpha=0.8)
axes[0].plot(range(6, 6 + len(media_movil)), media_movil,
             color='black', lw=2, label='Media móvil anual')
axes[0].set_title('Serie SAR(1)₁₂  (precipitación simulada)')
axes[0].set_xlabel('Mes'); axes[0].set_ylabel('X_t')
axes[0].legend(fontsize=9)

axes[1].bar(range(max_lag + 1), acf_sar, color='#00BCD4', alpha=0.75, width=0.6)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].axhline( 1.96/np.sqrt(n_meses), color='gray', lw=1, ls='--')
axes[1].axhline(-1.96/np.sqrt(n_meses), color='gray', lw=1, ls='--')
for lag_s in [12, 24, 36]:
    axes[1].axvline(lag_s, color='#E91E63', lw=1.5, ls=':', alpha=0.7)
axes[1].set_title('ACF — picos en lags múltiplos de 12')
axes[1].set_xlabel('Lag h'); axes[1].set_ylim(-1.1, 1.1)

# Distribución marginal
axes[2].hist(x_sar, bins=35, density=True, color='#00BCD4', alpha=0.6)
xx = np.linspace(x_sar.min(), x_sar.max(), 200)
sigma_teo = np.sqrt(1 / (1 - phi12**2))
axes[2].plot(xx, stats.norm.pdf(xx, 0, sigma_teo), 'r-', lw=2, label='N(0, σ²) teórica')
axes[2].set_title('Distribución marginal')
axes[2].set_xlabel('X_t'); axes[2].legend(fontsize=9)

fig.suptitle('Proceso estacionario con estructura estacional\n'
             'Ref: Montanari et al. (2000), Water Resources Research',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Módulo 2 — Cadenas de Markov en tiempo discreto

Una cadena de Markov es un proceso $\{X_n\}_{n \geq 0}$ con espacio de estados $\mathcal{S}$ que satisface:

$$P(X_{n+1} = j \mid X_n = i, X_{n-1}, \ldots, X_0) = P(X_{n+1} = j \mid X_n = i) = p_{ij}$$

La **propiedad de Markov** dice que el futuro depende del pasado solo a través del estado presente.

### Ejemplo 2.1 — Modelo climático de tres estados

Estados: Soleado (S), Nublado (N), Lluvioso (L).  
**Referencia:** Wilks, D.S. (2011). *Statistical Methods in the Atmospheric Sciences*. Academic Press, Cap. 8.

In [ ]:
# Matriz de transición (Wilks 2011, adaptada)
#           S     N     L
P_clima = np.array([
    [0.70, 0.20, 0.10],  # desde S
    [0.30, 0.40, 0.30],  # desde N
    [0.20, 0.35, 0.45],  # desde L
])
estados = ['Soleado', 'Nublado', 'Lluvioso']
colores_clima = ['#FFC107', '#90A4AE', '#1565C0']

def simular_cadena(P, n_pasos, estado_inicial=0):
    estados_sim = [estado_inicial]
    for _ in range(n_pasos - 1):
        actual = estados_sim[-1]
        siguiente = np.random.choice(len(P), p=P[actual])
        estados_sim.append(siguiente)
    return np.array(estados_sim)

# Distribución estacionaria: resolver π P = π, sum(π) = 1
eigvals, eigvecs = np.linalg.eig(P_clima.T)
idx_uno = np.argmin(np.abs(eigvals - 1))
pi = np.real(eigvecs[:, idx_uno])
pi = pi / pi.sum()

n_sim = 365
cadena = simular_cadena(P_clima, n_sim)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Trayectoria
for i, (nombre, color) in enumerate(zip(estados, colores_clima)):
    mask = cadena == i
    axes[0].scatter(np.where(mask)[0], cadena[mask], color=color, s=8,
                    alpha=0.8, label=nombre)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(estados)
axes[0].set_xlabel('Día'); axes[0].set_title('Trayectoria (365 días)')
axes[0].legend(fontsize=9)

# Convergencia a distribución estacionaria
n_pasos_conv = 20
mu0 = np.array([1.0, 0.0, 0.0])  # empezamos en Soleado
distribs = [mu0 @ np.linalg.matrix_power(P_clima, n) for n in range(n_pasos_conv)]
distribs = np.array(distribs)

for i, (nombre, color) in enumerate(zip(estados, colores_clima)):
    axes[1].plot(distribs[:, i], color=color, lw=2, marker='o', markersize=4,
                 label=nombre)
    axes[1].axhline(pi[i], color=color, lw=1, ls='--')
axes[1].set_xlabel('Pasos n')
axes[1].set_ylabel('P(X_n = j | X_0 = Soleado)')
axes[1].set_title('Convergencia a π  (líneas punteadas)')
axes[1].legend(fontsize=9)

# Distribución estacionaria
freq_empirica = np.array([np.mean(cadena == i) for i in range(3)])
x_bar = np.arange(3)
axes[2].bar(x_bar - 0.2, pi, width=0.35, color=[c for c in colores_clima],
            alpha=0.9, label='π teórica')
axes[2].bar(x_bar + 0.2, freq_empirica, width=0.35,
            color=[c for c in colores_clima], alpha=0.4, label='Frec. empírica')
axes[2].set_xticks(x_bar); axes[2].set_xticklabels(estados)
axes[2].set_title('Distribución estacionaria π')
axes[2].legend(fontsize=9)

print("Distribución estacionaria π:")
for e, p in zip(estados, pi):
    print(f"  {e}: {p:.4f}")

fig.suptitle('Cadena de Markov — Modelo climático de 3 estados\n'
             'Ref: Wilks (2011), Statistical Methods in the Atmospheric Sciences, Cap. 8',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo 2.2 — Ruina del jugador

Un jugador tiene $k$ pesos y juega contra un casino con capital infinito.  
En cada ronda gana 1 peso con probabilidad $p$ o pierde 1 con probabilidad $1-p$.  
El juego termina cuando llega a 0 (ruina) o a $N$ (meta).

La probabilidad de ruina partiendo de $k$ es:
$$P(\text{ruina} \mid X_0 = k) = \frac{(q/p)^k - (q/p)^N}{1 - (q/p)^N} \quad (p \neq q)$$

**Referencia:** Feller, W. (1968). *An Introduction to Probability Theory and Its Applications*, Vol. 1. Wiley, Cap. 14.

In [ ]:
def prob_ruina(k, N, p):
    q = 1 - p
    if abs(p - 0.5) < 1e-10:
        return 1 - k / N
    r = q / p
    return (r**k - r**N) / (1 - r**N)

def simular_ruina(k0, N, p, n_sim=1000):
    ruinas = 0
    for _ in range(n_sim):
        k = k0
        while 0 < k < N:
            k += 1 if np.random.random() < p else -1
        if k == 0:
            ruinas += 1
    return ruinas / n_sim

N = 20
k_vals_r = np.arange(1, N)
probs_sim = [0.45, 0.50, 0.55]
colores_r = ['#E91E63', '#607D8B', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for p, color in zip(probs_sim, colores_r):
    teo = [prob_ruina(k, N, p) for k in k_vals_r]
    sim = [simular_ruina(k, N, p, n_sim=500) for k in k_vals_r]
    axes[0].plot(k_vals_r, teo, color=color, lw=2.5, label=f'p={p} (teórica)')
    axes[0].scatter(k_vals_r, sim, color=color, s=25, alpha=0.5)

axes[0].set_xlabel('Capital inicial k')
axes[0].set_ylabel('P(ruina)')
axes[0].set_title(f'Probabilidad de ruina  (N={N})')
axes[0].legend(fontsize=9)

# Trayectorias
p_tray = 0.48
k0_tray = 10
for _ in range(8):
    tray = [k0_tray]
    while 0 < tray[-1] < N:
        tray.append(tray[-1] + (1 if np.random.random() < p_tray else -1))
    color_t = '#E91E63' if tray[-1] == 0 else '#4CAF50'
    axes[1].plot(tray, lw=1.2, alpha=0.7, color=color_t)

axes[1].axhline(0, color='black', lw=1.5, label='Ruina')
axes[1].axhline(N, color='#4CAF50', lw=1.5, ls='--', label=f'Meta N={N}')
axes[1].set_xlabel('Ronda'); axes[1].set_ylabel('Capital')
axes[1].set_title(f'Trayectorias  (p={p_tray}, k₀={k0_tray})')
axes[1].legend(fontsize=9)

fig.suptitle('Ruina del jugador — estados absorbentes en 0 y N\n'
             'Ref: Feller (1968), Probability Theory and Its Applications, Cap. 14',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo 2.3 — PageRank como distribución estacionaria

El algoritmo PageRank de Google modela la navegación web como una cadena de Markov:  
un usuario sigue links al azar con probabilidad $\alpha$ y salta a una página aleatoria con probabilidad $1-\alpha$.  
El ranking de cada página es su probabilidad en la distribución estacionaria.

$$P_{\text{PageRank}} = \alpha M + (1-\alpha) \frac{\mathbf{1}\mathbf{1}^T}{n}$$

**Referencia:** Brin, S. & Page, L. (1998). *The anatomy of a large-scale hypertextual web search engine*. Computer Networks, 30(1-7), 107–117.

In [ ]:
import matplotlib.patches as mpatches

# Grafo de 6 páginas (matriz de adyacencia)
# Página i apunta a página j si adj[i,j]=1
adj = np.array([
    [0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 1, 1],
    [1, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0],
])
n_pag = adj.shape[0]

# Matriz de transición de links (normalizar filas)
row_sums = adj.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
M = adj / row_sums

# PageRank con factor de amortiguamiento alpha
alpha = 0.85
P_pr = alpha * M + (1 - alpha) * np.ones((n_pag, n_pag)) / n_pag

# Distribución estacionaria por potencias
pi_pr = np.ones(n_pag) / n_pag
for _ in range(100):
    pi_pr = pi_pr @ P_pr

# Convergencia
distribs_pr = [np.ones(n_pag) / n_pag]
for _ in range(30):
    distribs_pr.append(distribs_pr[-1] @ P_pr)
distribs_pr = np.array(distribs_pr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PageRank final
colores_pr = plt.cm.RdYlGn(pi_pr / pi_pr.max())
bars = axes[0].bar(range(n_pag), pi_pr, color=colores_pr, edgecolor='white', lw=0.5)
axes[0].set_xticks(range(n_pag))
axes[0].set_xticklabels([f'Pág {i+1}' for i in range(n_pag)])
axes[0].set_ylabel('PageRank (π)')
axes[0].set_title(f'Distribución estacionaria = PageRank  (α={alpha})')
for bar, v in zip(bars, pi_pr):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.002,
                 f'{v:.3f}', ha='center', fontsize=9)

# Convergencia
for i in range(n_pag):
    axes[1].plot(distribs_pr[:, i], lw=2, label=f'Pág {i+1}',
                 color=plt.cm.tab10(i/n_pag))
    axes[1].axhline(pi_pr[i], lw=0.8, ls='--', color=plt.cm.tab10(i/n_pag))
axes[1].set_xlabel('Iteración'); axes[1].set_ylabel('π_i^(n)')
axes[1].set_title('Convergencia de la distribución')
axes[1].legend(fontsize=8, ncol=2)

fig.suptitle('PageRank como distribución estacionaria de una cadena de Markov\n'
             'Ref: Brin & Page (1998), Computer Networks 30(1-7)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Módulo 3 — Procesos de Markov en tiempo continuo y Proceso de Poisson

Un proceso de Markov en tiempo continuo $\{X_t\}_{t \geq 0}$ satisface la propiedad de Markov en tiempo continuo.  
Su dinámica infinitesimal está descripta por el **generador infinitesimal** $Q = (q_{ij})$:
- $q_{ij} \geq 0$ para $i \neq j$: tasa de salto de $i$ a $j$
- $q_{ii} = -\sum_{j \neq i} q_{ij}$: tasa total de salida de $i$

Las distribuciones satisfacen las **ecuaciones de Kolmogorov**:  
$$\frac{d}{dt} P(t) = P(t) Q \quad (\text{forward}) \qquad \frac{d}{dt} P(t) = Q P(t) \quad (\text{backward})$$

con solución $P(t) = e^{Qt}$ (exponencial matricial).

In [ ]:
# Proceso de nacimiento y muerte: cola M/M/1
# Estados: número de clientes en sistema  {0, 1, 2, ..., S_max}
# Tasa de llegada λ, tasa de servicio μ
# Referencia: Kleinrock, L. (1975). Queueing Systems Vol. 1. Wiley.

lam = 2.5   # llegadas por minuto
mu  = 3.0   # atenciones por minuto
S   = 8     # estados: 0..S

# Generador Q
Q = np.zeros((S+1, S+1))
for i in range(S+1):
    if i < S:
        Q[i, i+1] = lam
    if i > 0:
        Q[i, i-1] = mu
    Q[i, i] = -Q[i, :].sum()

# Distribución estacionaria analítica M/M/1
rho = lam / mu
pi_mm1 = np.array([(1 - rho) * rho**k for k in range(S+1)])
pi_mm1 /= pi_mm1.sum()  # truncada

# Evolución temporal: P(t) = exp(Qt)
t_vals = np.linspace(0, 5, 200)
p0 = np.zeros(S+1); p0[0] = 1.0  # empezamos vacíos
distribs_ct = np.array([p0 @ expm(Q * t) for t in t_vals])

# Simulación de trayectoria
def simular_mm1(lam, mu, T_max, estado_inicial=0):
    t, s = 0, estado_inicial
    tiempos, estados = [0], [s]
    while t < T_max:
        tasa_salida = (lam if s >= 0 else 0) + (mu if s > 0 else 0)
        if tasa_salida == 0:
            break
        dt = np.random.exponential(1 / tasa_salida)
        t += dt
        if t > T_max:
            break
        if s == 0:
            s += 1
        else:
            s += 1 if np.random.random() < lam / tasa_salida else -1
        tiempos.append(t)
        estados.append(s)
    return np.array(tiempos), np.array(estados)

t_tray, s_tray = simular_mm1(lam, mu, T_max=10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].step(t_tray, s_tray, where='post', color='#9C27B0', lw=1.5)
axes[0].set_xlabel('t (min)'); axes[0].set_ylabel('Clientes en sistema')
axes[0].set_title(f'Trayectoria M/M/1  (λ={lam}, μ={mu})')

for i in range(0, S+1, 2):
    axes[1].plot(t_vals, distribs_ct[:, i], lw=1.8, label=f'k={i}')
    axes[1].axhline(pi_mm1[i], lw=0.8, ls='--')
axes[1].set_xlabel('t'); axes[1].set_ylabel('P(X_t = k | X_0 = 0)')
axes[1].set_title('Convergencia a π  (ec. Kolmogorov forward)')
axes[1].legend(fontsize=8, ncol=2)

axes[2].bar(range(S+1), pi_mm1, color='#9C27B0', alpha=0.7)
axes[2].set_xlabel('k (clientes)'); axes[2].set_ylabel('π_k')
axes[2].set_title(f'Dist. estacionaria M/M/1  (ρ = λ/μ = {rho:.2f})')

fig.suptitle('Cola M/M/1 — proceso de nacimiento y muerte\n'
             'Ref: Kleinrock (1975), Queueing Systems Vol. 1, Wiley',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo 3.2 — Proceso de Poisson homogéneo y no homogéneo

El proceso de Poisson $\{N_t\}_{t \geq 0}$ es el proceso de conteo de eventos que ocurren de forma independiente y a tasa constante $\lambda$.  
Es un caso especial del proceso de nacimiento puro: $q_{k,k+1} = \lambda$.

**Poisson no homogéneo:** la tasa varía en el tiempo, $\lambda(t)$.  
El número esperado de eventos en $[0,t]$ es $\Lambda(t) = \int_0^t \lambda(s)\, ds$.

**Referencia:** Cox, D.R. & Lewis, P.A.W. (1966). *The Statistical Analysis of Series of Events*. Chapman & Hall.

In [ ]:
def simular_poisson_homogeneo(lam, T):
    """Tiempos de llegada de un PP homogéneo en [0,T]."""
    tiempos = []
    t = 0
    while True:
        t += np.random.exponential(1 / lam)
        if t > T:
            break
        tiempos.append(t)
    return np.array(tiempos)

def simular_poisson_no_homogeneo(lam_func, lam_max, T):
    """Thinning: simular PP no homogéneo por aceptación-rechazo."""
    candidatos = simular_poisson_homogeneo(lam_max, T)
    aceptados = [t for t in candidatos
                 if np.random.random() < lam_func(t) / lam_max]
    return np.array(aceptados)

T_sim = 20
lam_hom = 3.0

# Tasa no homogénea: modelo de llamadas telefónicas (pico al mediodía)
# λ(t) = 2 + 4*sin²(πt/24) — mayor tasa a t=12
lam_func = lambda t: 1.5 + 3 * np.sin(np.pi * t / T_sim)**2
lam_max  = 4.5

arr_hom  = simular_poisson_homogeneo(lam_hom, T_sim)
arr_nhom = simular_poisson_no_homogeneo(lam_func, lam_max, T_sim)

t_grid = np.linspace(0, T_sim, 300)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# PP homogéneo: proceso de conteo
conteo_hom = np.arange(len(arr_hom) + 1)
t_hom = np.concatenate([[0], arr_hom])
axes[0,0].step(t_hom, conteo_hom, where='post', color='#2196F3', lw=2)
axes[0,0].set_xlabel('t'); axes[0,0].set_ylabel('N(t)')
axes[0,0].set_title(f'PP homogéneo  λ={lam_hom}  →  {len(arr_hom)} eventos')

# Intervalos entre llegadas: deben ser Exp(λ)
if len(arr_hom) > 1:
    interarribos = np.diff(np.concatenate([[0], arr_hom]))
    axes[0,1].hist(interarribos, bins=20, density=True,
                   color='#2196F3', alpha=0.6, label='Empírico')
    xx = np.linspace(0, interarribos.max(), 200)
    axes[0,1].plot(xx, stats.expon.pdf(xx, scale=1/lam_hom),
                   'r-', lw=2, label=f'Exp({lam_hom})')
    axes[0,1].set_xlabel('Interarribo'); axes[0,1].set_ylabel('Densidad')
    axes[0,1].set_title('Interarribos ~ Exp(λ)')
    axes[0,1].legend(fontsize=9)

# PP no homogéneo: tasa y proceso de conteo
ax_tasa = axes[1,0].twinx()
conteo_nhom = np.arange(len(arr_nhom) + 1)
t_nhom = np.concatenate([[0], arr_nhom])
axes[1,0].step(t_nhom, conteo_nhom, where='post', color='#E91E63', lw=2, label='N(t)')
ax_tasa.plot(t_grid, lam_func(t_grid), color='#FF9800', lw=2, ls='--', label='λ(t)')
axes[1,0].set_xlabel('t'); axes[1,0].set_ylabel('N(t)', color='#E91E63')
ax_tasa.set_ylabel('λ(t)', color='#FF9800')
axes[1,0].set_title(f'PP no homogéneo  →  {len(arr_nhom)} eventos')

# Verificación: N(t) ~ Poisson(Λ(t))
n_rep = 1000
t_check = T_sim / 2
Lambda_t = np.trapz(lam_func(np.linspace(0, t_check, 500)),
                    np.linspace(0, t_check, 500))
counts = [len(simular_poisson_no_homogeneo(lam_func, lam_max, t_check))
          for _ in range(n_rep)]
k_max = max(counts)
axes[1,1].hist(counts, bins=range(k_max+2), density=True, align='left',
               color='#E91E63', alpha=0.6, label='Empírico')
k_vals_p = np.arange(k_max+1)
axes[1,1].plot(k_vals_p, stats.poisson.pmf(k_vals_p, Lambda_t),
               'ko-', markersize=5, lw=1.5, label=f'Poisson(Λ={Lambda_t:.1f})')
axes[1,1].set_xlabel(f'N(t={t_check})')
axes[1,1].set_title(f'Verificación: N(t) ~ Poisson(Λ(t))')
axes[1,1].legend(fontsize=9)

fig.suptitle('Proceso de Poisson homogéneo y no homogéneo\n'
             'Ref: Cox & Lewis (1966), The Statistical Analysis of Series of Events',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Módulo 4 — Movimiento Browniano

El **movimiento Browniano estándar** (proceso de Wiener) $\{W_t\}_{t \geq 0}$ satisface:
1. $W_0 = 0$ c.s.
2. Incrementos independientes: $W_t - W_s \perp \mathcal{F}_s$ para $t > s$
3. Incrementos estacionarios: $W_t - W_s \sim N(0, t-s)$
4. Trayectorias continuas c.s.

**Referencia:** Wiener, N. (1923). *Differential space*. Journal of Mathematics and Physics, 2, 131–174.

In [ ]:
# Construcción por límite de caminatas aleatorias
T_bm = 1.0
n_tray_bm = 5

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colores_bm = plt.cm.tab10(np.linspace(0, 0.8, n_tray_bm))

for n_pasos in [20, 100, 1000]:
    ax_idx = [20, 100, 1000].index(n_pasos)
    dt = T_bm / n_pasos
    t_bm = np.linspace(0, T_bm, n_pasos + 1)
    for i in range(n_tray_bm):
        inc = np.random.normal(0, np.sqrt(dt), n_pasos)
        W = np.concatenate([[0], np.cumsum(inc)])
        axes[ax_idx].plot(t_bm, W, lw=1.2, alpha=0.8, color=colores_bm[i])
    axes[ax_idx].set_xlabel('t')
    axes[ax_idx].set_ylabel('W_t')
    axes[ax_idx].set_title(f'n = {n_pasos} pasos  (Δt = {dt:.4f})')

fig.suptitle('Movimiento Browniano — construcción por caminatas aleatorias\n'
             'Ref: Wiener (1923), Journal of Mathematics and Physics',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo 4.2 — Variación cuadrática y no diferenciabilidad

Una de las propiedades más importantes del MB es que su **variación cuadrática** es determinista:
$$[W]_t = t \quad \text{c.s.}$$

Esto implica que las trayectorias tienen **variación infinita** y son **casi seguramente no diferenciables** en ningún punto.

**Referencia:** Karatzas & Shreve (1991), Teorema 2.9.18.

In [ ]:
def variacion_cuadratica(W, t_vals):
    """Variación cuadrática empírica de una trayectoria."""
    return np.cumsum(np.diff(W)**2)

def variacion_primera(W):
    """Variación de primer orden."""
    return np.cumsum(np.abs(np.diff(W)))

n_bm = 5000
T_bm2 = 1.0
dt_bm = T_bm2 / n_bm
t_bm2 = np.linspace(0, T_bm2, n_bm + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

n_tray_var = 4
for i in range(n_tray_var):
    inc = np.random.normal(0, np.sqrt(dt_bm), n_bm)
    W_i = np.concatenate([[0], np.cumsum(inc)])
    vc  = variacion_cuadratica(W_i, t_bm2)
    vp  = variacion_primera(W_i)
    axes[0].plot(t_bm2, W_i, lw=1, alpha=0.7)
    axes[1].plot(t_bm2[1:], vc, lw=1.5, alpha=0.8)
    axes[2].plot(t_bm2[1:], vp, lw=1.5, alpha=0.8)

axes[1].plot(t_bm2, t_bm2, 'k--', lw=2.5, label='[W]_t = t  (teórica)')
axes[1].legend(fontsize=9)

axes[0].set_title('Trayectorias W_t')
axes[0].set_xlabel('t'); axes[0].set_ylabel('W_t')
axes[1].set_title('Variación cuadrática  [W]_t → t')
axes[1].set_xlabel('t')
axes[2].set_title('Variación de primer orden → ∞')
axes[2].set_xlabel('t')

fig.suptitle('Variación cuadrática y variación de primer orden del MB\n'
             'Ref: Karatzas & Shreve (1991), Teorema 2.9.18',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejemplo 4.3 — MB con deriva y movimiento Browniano geométrico

**MB con deriva:** $X_t = \mu t + \sigma W_t$  
**MB geométrico (GBM):** $S_t = S_0 \exp\left((\mu - \frac{\sigma^2}{2})t + \sigma W_t\right)$

El GBM es el modelo clásico de precios de activos financieros (Black & Scholes, 1973).

**Referencia:** Black, F. & Scholes, M. (1973). *The pricing of options and corporate liabilities*. Journal of Political Economy, 81(3), 637–654.

In [ ]:
mu_bm  = 0.1
sigma_bm = 0.3
S0 = 100.0
T_fin = 1.0
n_bm3 = 252  # días hábiles
dt3 = T_fin / n_bm3
t3 = np.linspace(0, T_fin, n_bm3 + 1)
n_tray3 = 8

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

S_finales = []
for i in range(n_tray3):
    inc = np.random.normal(0, np.sqrt(dt3), n_bm3)
    W_i = np.concatenate([[0], np.cumsum(inc)])

    # MB con deriva
    X_i = mu_bm * t3 + sigma_bm * W_i
    axes[0].plot(t3, X_i, lw=1.2, alpha=0.75)

    # GBM
    S_i = S0 * np.exp((mu_bm - 0.5 * sigma_bm**2) * t3 + sigma_bm * W_i)
    axes[1].plot(t3, S_i, lw=1.2, alpha=0.75)
    S_finales.append(S_i[-1])

axes[0].axline((0,0), slope=mu_bm, color='black', lw=2, ls='--', label=f'Deriva μ={mu_bm}')
axes[0].set_title(f'MB con deriva  μ={mu_bm}, σ={sigma_bm}')
axes[0].set_xlabel('t'); axes[0].set_ylabel('X_t')
axes[0].legend(fontsize=9)

# Precio esperado
axes[1].plot(t3, S0 * np.exp(mu_bm * t3), 'k--', lw=2,
             label=f'E[S_t] = S₀ e^{{μt}}')
axes[1].set_title(f'MB Geométrico  (Black-Scholes)\nS₀={S0}, μ={mu_bm}, σ={sigma_bm}')
axes[1].set_xlabel('t'); axes[1].set_ylabel('S_t')
axes[1].legend(fontsize=9)

# Distribución de S_T: log-normal
n_mc = 10000
W_T = np.random.normal(0, np.sqrt(T_fin), n_mc)
S_T = S0 * np.exp((mu_bm - 0.5*sigma_bm**2)*T_fin + sigma_bm*W_T)
axes[2].hist(S_T, bins=60, density=True, color='#9C27B0', alpha=0.6, label='Empírico')
xx = np.linspace(S_T.min(), np.percentile(S_T, 99), 300)
mu_ln  = np.log(S0) + (mu_bm - 0.5*sigma_bm**2)*T_fin
sig_ln = sigma_bm * np.sqrt(T_fin)
axes[2].plot(xx, stats.lognorm.pdf(xx, s=sig_ln, scale=np.exp(mu_ln)),
             'r-', lw=2, label='Log-normal teórica')
axes[2].set_xlabel('S_T'); axes[2].set_ylabel('Densidad')
axes[2].set_title('Distribución de S_T  (log-normal)')
axes[2].legend(fontsize=9)

fig.suptitle('MB con deriva y Movimiento Browniano Geométrico\n'
             'Ref: Black & Scholes (1973), Journal of Political Economy 81(3)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Módulo 5 — Martingalas

Las martingalas modelan un tipo de dependencia temporal radicalmente diferente a los procesos Markovianos: **el pasado completo importa**, no solo el estado actual, pero la mejor predicción del futuro dada toda la información disponible es simplemente el valor presente.

Formalmente: $\{M_t, \mathcal{F}_t\}$ es una martingala si $E[M_t \mid \mathcal{F}_s] = M_s$ para $t > s$.  
Son la formalización de la noción de **juego justo** y de **precio eficiente** en mercados financieros.

**Referencia:** Doob, J.L. (1953). *Stochastic Processes*. Wiley.

In [ ]:
# Tres ejemplos clásicos de martingalas
n_mart = 500
n_tray_m = 6

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colores_m = plt.cm.tab10(np.linspace(0, 0.8, n_tray_m))

for i in range(n_tray_m):
    xi = np.random.choice([-1, 1], size=n_mart)
    S  = np.concatenate([[0], np.cumsum(xi)])           # 1. Caminata simétrica
    S2 = S**2 - np.arange(n_mart + 1)                  # 2. S_n^2 - n
    E  = np.exp(0.5 * S - 0.5 * np.log(np.cosh(0.5)) * np.arange(n_mart+1))  # 3. Exponencial

    axes[0].plot(S,  lw=1, alpha=0.7, color=colores_m[i])
    axes[1].plot(S2, lw=1, alpha=0.7, color=colores_m[i])
    axes[2].plot(E,  lw=1, alpha=0.7, color=colores_m[i])

for ax, titulo, ref in zip(axes,
    ['$S_n$ = caminata aleatoria simétrica\n$E[S_n|\\mathcal{F}_{n-1}] = S_{n-1}$',
     '$S_n^2 - n$  (martingala de cuadrado)\n$E[S_n^2 - n|\\mathcal{F}_{n-1}] = S_{n-1}^2 - (n-1)$',
     'Martingala exponencial\n$E[e^{\\theta S_n}/\\phi(\\theta)^n|\\mathcal{F}_{n-1}] = M_{n-1}$'],
    ['', '', '']):
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_title(titulo, fontsize=9)
    ax.set_xlabel('n')

fig.suptitle('Tres martingalas clásicas\n'
             'Ref: Doob (1953), Stochastic Processes, Wiley',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Teorema de parada opcional: E[S_τ] = E[S_0] = 0
# τ = tiempo de ruina o meta en la caminata simétrica
# Por OST: E[S_τ] = 0, lo que implica P(llegar a N antes de 0) = k/N

N_ost = 10
n_rep_ost = 3000
k_vals_ost = range(1, N_ost)

prob_meta_sim = []
prob_meta_teo = []
E_tau_sim = []

for k0 in k_vals_ost:
    llega_meta = 0
    tiempos_parada = []
    for _ in range(n_rep_ost):
        k = k0
        t_stop = 0
        while 0 < k < N_ost:
            k += np.random.choice([-1, 1])
            t_stop += 1
        if k == N_ost:
            llega_meta += 1
        tiempos_parada.append(t_stop)
    prob_meta_sim.append(llega_meta / n_rep_ost)
    prob_meta_teo.append(k0 / N_ost)          # por OST
    E_tau_sim.append(np.mean(tiempos_parada))

E_tau_teo = [k * (N_ost - k) for k in k_vals_ost]  # E[τ] = k(N-k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(k_vals_ost, prob_meta_teo, 'k--', lw=2, label='k/N  (OST teórica)')
axes[0].scatter(k_vals_ost, prob_meta_sim, color='#2196F3', s=60, zorder=5, label='Simulación')
axes[0].set_xlabel('Capital inicial k')
axes[0].set_ylabel('P(llegar a N antes de 0)')
axes[0].set_title(f'Teorema de Parada Opcional  (N={N_ost})')
axes[0].legend()

axes[1].plot(k_vals_ost, E_tau_teo, 'k--', lw=2, label='k(N-k)  (teórico)')
axes[1].scatter(k_vals_ost, E_tau_sim, color='#E91E63', s=60, zorder=5, label='Simulación')
axes[1].set_xlabel('Capital inicial k')
axes[1].set_ylabel('E[τ]  (pasos esperados)')
axes[1].set_title('Tiempo esperado de parada')
axes[1].legend()

fig.suptitle('Teorema de Parada Opcional — caminata aleatoria simétrica\n'
             'Ref: Durrett (2019), Probability: Theory and Examples, Cap. 5',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Módulo 6 — Cálculo estocástico

El cálculo estocástico extiende el análisis clásico a trayectorias no diferenciables como las del MB.  
A diferencia de los procesos anteriores, aquí el objeto central no es la distribución de estados sino la **dinámica infinitesimal de una trayectoria**, descripta mediante ecuaciones diferenciales estocásticas (EDEs).

La herramienta clave es la **fórmula de Itô**: si $X_t$ satisface $dX_t = \mu_t dt + \sigma_t dW_t$ y $f$ es suficientemente suave:

$$df(X_t) = f'(X_t)\, dX_t + \frac{1}{2} f''(X_t)\, \sigma_t^2\, dt$$

El término $\frac{1}{2} f'' \sigma^2 dt$ es la **corrección de Itô**, ausente en el cálculo clásico, y surge de la variación cuadrática no nula del MB.

**Referencia:** Itô, K. (1951). *On stochastic differential equations*. Memoirs of the American Mathematical Society, 4.

In [ ]:
# Verificación numérica de la fórmula de Itô
# f(W_t) = W_t^2  →  d(W_t^2) = 2 W_t dW_t + dt
# Integrando: W_t^2 = 2 ∫_0^t W_s dW_s + t
# Por lo tanto: ∫_0^t W_s dW_s = (W_t^2 - t) / 2

n_ito = 10000
T_ito = 1.0
dt_ito = T_ito / n_ito
t_ito = np.linspace(0, T_ito, n_ito + 1)

n_rep_ito = 500
integral_ito_vals = []
formula_vals = []

for _ in range(n_rep_ito):
    dW = np.random.normal(0, np.sqrt(dt_ito), n_ito)
    W  = np.concatenate([[0], np.cumsum(dW)])

    # Integral de Itô numérica: suma de W_{t_k} * ΔW_k  (punto izquierdo)
    integral_ito = np.sum(W[:-1] * dW)
    formula_ito  = (W[-1]**2 - T_ito) / 2

    integral_ito_vals.append(integral_ito)
    formula_vals.append(formula_ito)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Scatter: integral numérica vs fórmula
axes[0].scatter(formula_vals, integral_ito_vals, alpha=0.3, s=15, color='#9C27B0')
lims = [min(formula_vals+integral_ito_vals), max(formula_vals+integral_ito_vals)]
axes[0].plot(lims, lims, 'r--', lw=2, label='y = x')
axes[0].set_xlabel('$(W_T^2 - T)/2$  (fórmula de Itô)')
axes[0].set_ylabel('$\\int_0^T W_s\\, dW_s$  (numérica)')
axes[0].set_title('Verificación: fórmula de Itô')
axes[0].legend()

# Distribución de la integral
axes[1].hist(integral_ito_vals, bins=40, density=True,
             color='#9C27B0', alpha=0.6, label='Integral de Itô')
axes[1].set_xlabel('$\\int_0^1 W_s\\, dW_s$')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Distribución de $\\int_0^1 W_s\\, dW_s = (W_1^2 - 1)/2$')

# EDE: Ornstein-Uhlenbeck  dX_t = -θ X_t dt + σ dW_t
# Solución: X_t = X_0 e^{-θt} + σ ∫_0^t e^{-θ(t-s)} dW_s
theta_ou = 2.0
sigma_ou = 1.0
x0_ou    = 3.0
n_ou = 2000; dt_ou = T_ito / n_ou
t_ou = np.linspace(0, T_ito * 3, n_ou * 3 + 1)

for _ in range(6):
    dW_ou = np.random.normal(0, np.sqrt(dt_ou), len(t_ou) - 1)
    X_ou  = np.zeros(len(t_ou))
    X_ou[0] = x0_ou
    for j in range(1, len(t_ou)):
        X_ou[j] = X_ou[j-1] - theta_ou * X_ou[j-1] * dt_ou + sigma_ou * dW_ou[j-1]
    axes[2].plot(t_ou, X_ou, lw=1, alpha=0.7)

sigma_est = sigma_ou / np.sqrt(2 * theta_ou)
axes[2].axhline(0, color='black', lw=1.5, ls='--', label='Media estacionaria')
axes[2].axhline( 2*sigma_est, color='gray', lw=1, ls=':', label='±2σ∞')
axes[2].axhline(-2*sigma_est, color='gray', lw=1, ls=':')
axes[2].set_xlabel('t'); axes[2].set_ylabel('X_t')
axes[2].set_title(f'Ornstein-Uhlenbeck  θ={theta_ou}, σ={sigma_ou}\n'
                  f'$dX_t = -\\theta X_t dt + \\sigma dW_t$')
axes[2].legend(fontsize=9)

fig.suptitle('Cálculo estocástico — fórmula de Itô y EDE Ornstein-Uhlenbeck\n'
             'Ref: Itô (1951), Memoirs AMS 4  |  Uhlenbeck & Ornstein (1930), Phys Rev 36',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparación: Itô vs Stratonovich en una EDE simple
# dX_t = X_t dW_t
# Solución Itô:         X_t = X_0 exp(W_t - t/2)
# Solución Stratonovich: X_t = X_0 exp(W_t)
# La diferencia es exactamente la corrección de Itô: -t/2

n_comp = 5000
T_comp = 1.0
dt_comp = T_comp / n_comp
n_tray_comp = 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
t_comp = np.linspace(0, T_comp, n_comp + 1)

for _ in range(n_tray_comp):
    dW_c = np.random.normal(0, np.sqrt(dt_comp), n_comp)
    W_c  = np.concatenate([[0], np.cumsum(dW_c)])
    X_ito = np.exp(W_c - 0.5 * t_comp)    # solución Itô
    X_str = np.exp(W_c)                    # solución Stratonovich
    axes[0].plot(t_comp, X_ito, lw=1.2, alpha=0.75)
    axes[1].plot(t_comp, X_str, lw=1.2, alpha=0.75)

# Medias esperadas
axes[0].plot(t_comp, np.ones_like(t_comp), 'k--', lw=2,
             label='$E[X_t^{\\text{Itô}}] = X_0 = 1$')
axes[1].plot(t_comp, np.exp(0.5 * t_comp), 'k--', lw=2,
             label='$E[X_t^{\\text{Str}}] = e^{t/2}$')

for ax, titulo in zip(axes, ['Integral de Itô', 'Integral de Stratonovich']):
    ax.set_xlabel('t'); ax.set_ylabel('X_t')
    ax.set_title(f'{titulo}\n$dX_t = X_t\\, dW_t$')
    ax.legend(fontsize=9)

fig.suptitle('Itô vs Stratonovich — la corrección importa\n'
             'Ref: Øksendal (2003), Stochastic Differential Equations, Springer, Cap. 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()